# Custom OpenTelemetry metrics demo

Demonstrates custom OpenTelemetry metrics collection using a basic Strands agent hosted in Amazon Bedrock AgentCore Runtime. Here is a checklist of essentials:

* ✅ __IMPORTANT:__ Ensure metrics cache is flushed (`force_flush()`) before micro VM termination
* ✅ Python ADOT auto-instrumentation library is included in requirements.txt
* ✅ Add `opentelemetry-instrument` to the Docker CMD launching the agent
* ✅ Verify the AgentCore Runtime IAM execution policy includes permissions for CloudWatch Logs, X-Ray, and Metrics


<details>
  <summary><span style="color: #D97706;weight: bold;">Sample observability IAM permissions for Runtime IAM execution policy</span></summary>

**Reference:** https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html


> **Note:** Implement the principle of least privilege and update IAM resource restrictions and conditions as needed for your project.

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "logs:DescribeLogStreams",
        "logs:CreateLogGroup"
      ],
      "Resource": [
        "arn:aws:logs:us-east-1:123456789012:log-group:*"
      ]
    },
    {
      "Effect": "Allow",
      "Action": [
        "logs:DescribeLogGroups"
      ],
      "Resource": [
        "arn:aws:logs:us-east-1:123456789012:log-group:*"
      ]
    },
    {
      "Effect": "Allow",
      "Action": [
        "logs:CreateLogStream",
        "logs:PutLogEvents"
      ],
      "Resource": [
        "arn:aws:logs:us-east-1:123456789012:log-group:*:log-stream:*"
      ]
    },
    {
      "Effect": "Allow",
      "Action": [
        "xray:PutTraceSegments",
        "xray:PutTelemetryRecords",
        "xray:GetSamplingRules",
        "xray:GetSamplingTargets"
      ],
      "Resource": [
        "*"
      ]
    },
    {
      "Effect": "Allow",
      "Resource": "*",
      "Action": "cloudwatch:PutMetricData"
    }
  ]
}
```
</details>


## Tutorial step-by-step

#### Setup python libraries

In [ ]:
# Agentcore starter toolkit (deprecated) for deploying the agent
%pip install -r requirements-dev.txt --quiet

# [Optional] Agent dependent libraries for local development
%pip install -r requirements.txt --quiet

#### Create Strands agent
The following agent code shows how Strands telemetry is initialized and custom OTEL metric is used to record agent requests.

In [ ]:
%%writefile otel_metrics.py
"""Strands OTEL custom metrics demo"""
import os
from strands import Agent
from strands.models import BedrockModel
from opentelemetry import metrics
from bedrock_agentcore.runtime import BedrockAgentCoreApp


MODEL_ID = os.getenv("BEDROCK_MODEL_ID")
REGION = os.getenv("AWS_REGION", "us-east-1")

# Create OTEL metrics meter
meter_provider = metrics.get_meter_provider()
meter = metrics.get_meter("otel-metrics-demo-meter", "1.0.0")
request_histogram = meter.create_histogram(
    name="agent_requests",
    description="Total number of invocation requests",
    unit="requests"
)
request_counter = meter.create_counter(
    name="agent_requests_total",
    description="Total number of invocation requests",
    unit="1"
)

# Initialize agent and bedrock application wrapper
agent = Agent(model=BedrockModel(model_id=MODEL_ID))
app = BedrockAgentCoreApp()


@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)

    request_histogram.record(1, {"agent.name": "otel_metrics_demo"})
    request_counter.add(1, {"tenant.id": "otel_demo_bu"})
    response = agent(user_input)
    
    # Required: to ensure the metrics are flushed before runtime termination
    meter_provider.force_flush()
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


#### Configure AgentCore Runtime for deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agent_name = "otel_metrics_demo"
response = agentcore_runtime.configure(
    entrypoint="otel_metrics.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region="us-east-1",
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
)
response

### Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "BEDROCK_MODEL_ID": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        "AWS_REGION": "us-east-1",
    }
)
launch_result

### Check Deployment Status

Wait for the runtime and memory to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload. Invoke multiple times.

In [ ]:
!agentcore invoke '{"prompt": "What is the capital of usa?"}'

### View custom OTEL metrics in CloudWatch logs and metrics explorer

> Note: Allow few minutes for the metrics to be coalesced and written into CloudWatch metrics and logs

#### OTEL custom metrics in CloudWatch Logs
Go to the Runtime log group > `otel-rt-logs` log stream to see the metrics recorded as log entries. 
Note down the `namespace` , `metrics` and `dimensions` in the log.This will be useful to locate the metric 
in CloudWatch metrics explorer.

<img src="./img/emf-cw-log.png" width="50%" />

#### OTEL metrics in CloudWatch metrics explorer
To view custom metrics in CloudWatch metrics explorer, go to "All metrics" > Custom namespaces > metric namespace (`bedrock-agentcore`) > metric dimension (`agent.name`). Select metric for plotting.

1/ Locate custom namespace.  
<img src="./img/cw-metrics-ns.png" width="50%" />

2/ Find metric dimension.  
<img src="./img/cw-metrics-dim.png" width="50%" />

3/ Plot the metric graph.  
<img src="./img/cw-metrics-graph.png" width="50%" />


## Cleanup instructions

Don't forget to cleanup any resources created. Remove `--dry-run` for actual deletion.

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run